# STEP 27: Unseen PEOPLE, not only unseen conversations.
NoXi metadata shows that two participants appear in two of our sessions
(person 12: sessions 001 and 021; person 44: sessions 028 and 030). With the
earlier folds the same person could be in training and in test.
Here every split keeps all sessions of a person together (src/splits.py):
folds  : 3 person-disjoint folds (GroupKFold over person groups)
within : leave one session out, ALSO removing sessions that share a person
cross  : train language A, test language B (no person is in two languages, unchanged)
Each model is scored both ways, so the size of the leak is visible.
Models: Gradient Boosting (same settings as steps 14/23/24) on
voice | face | voice + face | voice + face + context | eGeMAPS + w2v-BERT + VideoMAE + context (best)
Input:  data/features/noxi_engagement.parquet + pretrained_<name>.npy + data/noxi_metadata.xlsx
Output: reports/person_disjoint_results.csv

*Notebook version of `src/27_person_disjoint.py`: same code, split into cells. Running it does exactly what the script does.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os, sys
if os.getcwd().endswith("notebooks"):
    os.chdir("..")              # work from the project folder, like the scripts
sys.path.insert(0, "src")       # so helper files in src/ (e.g. splits.py) can be imported

Imports and settings.

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from splits import person_folds, same_person

FEAT = os.path.join("data", "features", "noxi_engagement.parquet")
META = [f"face_{i}" for i in range(5)]
CONTEXT = [5, 10]
OUT = os.path.join("reports", "person_disjoint_results.csv")

Function `ccc`.

In [ ]:
def ccc(yt, yp):
    yt, yp = np.asarray(yt), np.asarray(yp)
    cov = ((yt - yt.mean()) * (yp - yp.mean())).mean()
    return (2 * cov) / (yt.var() + yp.var() + (yt.mean() - yp.mean()) ** 2 + 1e-9)

Function `with_context`.

In [ ]:
def with_context(X, groups):
    frame = pd.DataFrame(X)
    return np.hstack([X] + [frame.groupby(groups).transform(lambda g: g.rolling(k, min_periods=1).mean()).to_numpy("float32")
                            for k in CONTEXT])

Load the data.

In [ ]:
df = pd.read_parquet(FEAT)
voice = [c for c in df.columns if c.startswith("voice_")]
face = [c for c in df.columns if c.startswith("face_") and c not in META]
y = df["engagement"].to_numpy("float32")
speaker = (df["session"] + "_" + df["role"]).to_numpy()
lang, session = df["language"].to_numpy(), df["session"].to_numpy()
langs = sorted(df["language"].unique())
sess = df.groupby("language")["session"].unique().apply(sorted)

old_folds = [[s[k] for s in sess] for k in range(3)]
new_folds = person_folds(df, 3)
mates = same_person(df["session"].unique())
print(f"{len(df)} windows\nold folds (conversation-disjoint): {old_folds}\nnew folds (person-disjoint):       {new_folds}")
print("sessions sharing a person:", {s: m for s, m in mates.items() if len(m) > 1}, flush=True)

own = df[voice + face].fillna(0).to_numpy("float32")
fused = np.hstack([df[voice].fillna(0).to_numpy("float32")] +
                  [np.load(os.path.join("data", "features", f"pretrained_{p}.npy")) for p in ("w2vbert2", "videomae")])
INPUTS = {
    "voice": df[voice].fillna(0).to_numpy("float32"),
    "face": df[face].fillna(0).to_numpy("float32"),
    "voice + face": own,
    "voice + face + context": with_context(own, speaker),
    "fusion + context (best)": with_context(fused, speaker),
}

Function `gb`.

In [ ]:
def gb(X, tr, te):
    m = make_pipeline(StandardScaler(), HistGradientBoostingRegressor(max_iter=300, random_state=42))
    return m.fit(X[tr], y[tr]).predict(X[te])

Set up for the next step.

In [ ]:
rows = []

Main loop.

In [ ]:
for name, X in INPUTS.items():
    r = dict(input=name)
    for tag, folds in (("old", old_folds), ("new", new_folds)):
        sc = [ccc(y[np.isin(session, t)], gb(X, ~np.isin(session, t), np.isin(session, t))) for t in folds]
        r[f"ccc_{tag}"], r[f"std_{tag}"] = np.mean(sc), np.std(sc, ddof=1)
    w_old, w_new = [], []
    for l in langs:
        for s in sess[l]:
            w_old.append(ccc(y[session == s], gb(X, (lang == l) & (session != s), session == s)))
            w_new.append(ccc(y[session == s], gb(X, (lang == l) & ~np.isin(session, mates[s]), session == s)))
    cross = [ccc(y[lang == b], gb(X, lang == a, lang == b)) for a, b in itertools.permutations(langs, 2)]
    c = np.mean(cross)
    r.update(within_old=np.mean(w_old), within_new=np.mean(w_new), cross=c,
             drop_old=100 * (np.mean(w_old) - c) / np.mean(w_old), drop_new=100 * (np.mean(w_new) - c) / np.mean(w_new))
    rows.append(r)
    pd.DataFrame(rows).to_csv(OUT, index=False)
    print(f"  {name:26s} folds {r['ccc_old']:.3f} -> {r['ccc_new']:.3f} (+-{r['std_new']:.3f}) | within {r['within_old']:.3f} -> "
          f"{r['within_new']:.3f} | cross {c:.3f} | drop {r['drop_old']:.0f}% -> {r['drop_new']:.0f}%", flush=True)

Show the results.

In [ ]:
print("\n=========== UNSEEN CONVERSATIONS (old) vs UNSEEN PEOPLE (new), Gradient Boosting ===========")
print(f"{'input':26s} {'folds old':>9s} {'folds new':>15s} {'within old':>10s} {'within new':>10s} {'cross':>6s} {'drop old':>8s} {'drop new':>8s}")

Main loop.

In [ ]:
for r in rows:
    print(f"{r['input']:26s} {r['ccc_old']:9.3f} {r['ccc_new']:7.3f} +-{r['std_new']:.3f} {r['within_old']:10.3f} "
          f"{r['within_new']:10.3f} {r['cross']:6.3f} {r['drop_old']:7.0f}% {r['drop_new']:7.0f}%")

Show the results.

In [ ]:
print(f"Saved -> {OUT}")